<a href="https://colab.research.google.com/github/VictoriaAzevedo/credit-score-analysis/blob/main/Projeto_Final_Credit_Score_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Projeto esenvolvido como parte do desafio final do Bootcamp de Analista de Dados da Re/Start tem como objetivo de analisar o perfil  dos clientes e identificar fatores associados à classificação do Score de Crédito.**

### **Importação das Bibliotecas**

In [29]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

### **Carregamento da Base**

In [21]:
df = pd.read_csv("train.csv")

/tmp/ipykernel_2151/2292805398.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("train.csv")


### **Análise Inicial da Base**

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  object 
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  object 
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  object 
 13  Type_of_Loan              88592 non-null   ob

In [34]:
resumo = pd.DataFrame({
    "Indicador": [
        "Total de registros",
        "Total de variáveis",
        "Variáveis categóricas (object)",
        "Variáveis inteiras (int64)",
        "Variáveis decimais (float64)",
        "Colunas com valores ausentes"
    ],
    "Valor": [
        df.shape[0],
        df.shape[1],
        (df.dtypes == "object").sum(),
        (df.dtypes == "int64").sum(),
        (df.dtypes == "float64").sum(),
        df.isnull().any().sum()
    ]
})

resumo

,Indicador,Valor
0,Total de registros,100000
1,Total de variáveis,28
2,Variáveis categóricas (object),20
3,Variáveis inteiras (int64),4
4,Variáveis decimais (float64),4
5,Colunas com valores ausentes,8


In [32]:
df.describe()

,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Delay_from_due_date,Num_Credit_Inquiries,Credit_Utilization_Ratio,Total_EMI_per_month
count,84998.000000,100000.000000,100000.00000,100000.000000,100000.000000,98035.000000,100000.000000,100000.000000
mean,4194.170850,17.091280,22.47443,72.466040,21.068780,27.754251,32.285173,1403.118217
std,3183.686167,117.404834,129.05741,466.422621,14.860104,193.177339,5.116875,8306.041270
min,303.645417,-1.000000,0.00000,1.000000,-5.000000,0.000000,20.000000,0.000000
25%,1625.568229,3.000000,4.00000,8.000000,10.000000,3.000000,28.052567,30.306660
50%,3093.745000,6.000000,5.00000,13.000000,18.000000,6.000000,32.305784,69.249473
75%,5957.448333,7.000000,7.00000,20.000000,28.000000,9.000000,36.496663,161.224249
max,15204.633333,1798.000000,1499.00000,5797.000000,67.000000,2597.000000,50.000000,82331.000000


In [33]:
df.describe(include='object')

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Num_of_Loan,Type_of_Loan,Num_of_Delayed_Payment,Changed_Credit_Limit,Credit_Mix,Outstanding_Debt,Credit_History_Age,Payment_of_Min_Amount,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
count,100000,100000,100000,90015,100000,100000,100000,100000,100000,88592,92998,100000,100000,100000,90970,100000,95521,100000,98800,100000
unique,100000,12500,8,10139,1788,12501,16,18940,434,6260,749,4384,4,13178,404,3,91049,7,98792,3
top,0x25fd5,CUS_0x942c,January,Stevex,38,#F%$D@*&8,_______,36585.12,3,Not Specified,19,_,Standard,1360.45,15 Years and 11 Months,Yes,__10000__,Low_spent_Small_value_payments,__-333333333333333333333333333__,Standard
freq,1,8,12500,44,2833,5572,7062,16,14386,1408,5327,2091,36479,24,446,52326,4305,25513,9,53174


Principais observações da estrutura dos dados

- O conjunto de dados possui **100.000 registros** distribuídos em **28 variáveis**, contendo informações cadastrais, financeiras e comportamentais dos clientes.

- Foram identificadas **oito variáveis com valores ausentes**, que serão tratadas durante a etapa de preparação dos dados (`Name`, `Monthly_Inhand_Salary`, `Type_of_Loan`, `Num_of_Delayed_Payment`, `Num_Credit_Inquiries`, `Credit_History_Age`, `Amount_invested_monthly` e `Monthly_Balance`).

- A base apresenta predominância de variáveis categóricas (`object`), enquanto apenas oito variáveis encontram-se em formato numérico (`int64` e `float64`).

- Algumas variáveis que representam valores numéricos encontram-se armazenadas como texto (`object`), indicando a necessidade de conversão para o tipo adequado antes da realização das análises.

- Foram identificados valores mínimos negativos em variáveis como `Num_Bank_Accounts` e `Delay_from_due_date`, que aparentam ser inconsistentes e serão investigados durante a etapa de limpeza dos dados.

- Algumas variáveis apresentam valores máximos bastante elevados, indicando possíveis valores extremos (outliers) ou inconsistências que deverão ser avaliadas durante a preparação dos dados.

- O conjunto de dados reúne informações cadastrais, financeiras e comportamentais, permitindo investigar diferentes fatores associados à classificação do score de crédito.

Conclusão da etapa

A análise inicial permitiu compreender a estrutura do conjunto de dados e identificar os principais desafios para as próximas etapas, especialmente relacionados ao tratamento de valores ausentes, conversão dos tipos de dados e investigação de possíveis inconsistências. A diversidade de informações disponíveis permitirá analisar a relação entre características financeiras e comportamentais dos clientes e sua classificação de score de crédito.

In [38]:
faltantes = pd.DataFrame({'colunas':df.columns,
                      'tipo':df.dtypes,
                      'Qtde valores NaN':df.isna().sum(),
                      '% valores NaN':(df.isna().sum()/df.shape[0])*100,
                      'valores únicos por feature':df.nunique()})
faltantes = faltantes.reset_index()
faltantes

,index,colunas,tipo,Qtde valores NaN,% valores NaN,valores únicos por feature
0,ID,ID,object,0,0.000,100000
1,Customer_ID,Customer_ID,object,0,0.000,12500
2,Month,Month,object,0,0.000,8
3,Name,Name,object,9985,9.985,10139
4,Age,Age,object,0,0.000,1788
5,SSN,SSN,object,0,0.000,12501
6,Occupation,Occupation,object,0,0.000,16
7,Annual_Income,Annual_Income,object,0,0.000,18940
8,Monthly_Inhand_Salary,Monthly_Inhand_Salary,float64,15002,15.002,13235
9,Num_Bank_Accounts,Num_Bank_Accounts,int64,0,0.000,943


### **Integridade e unicidade dos registros**

In [50]:
resumo_duplicidade = pd.DataFrame({
    "Indicador": [
        "Registros por ID",
        "Registros duplicados",
        "Duplicidade Cliente + Mês",
        "Meses analisados"
    ],
    "Valor": [
        df["Customer_ID"].nunique(),
        df.duplicated().sum(),
        df.duplicated(subset=["Customer_ID","Month"]).sum(),
        df["Month"].nunique()
    ]
})

resumo_duplicidade

,Indicador,Valor
0,Registros por ID,12500
1,Registros duplicados,0
2,Duplicidade Cliente + Mês,0
3,Meses analisados,8


In [51]:
df["Month"].unique()

array(['January', 'February', 'March', 'April', 'May', 'June', 'July',
       'August'], dtype=object)

Observações

Não foram identificados registros completamente duplicados, ou seja, nenhuma linha apresenta exatamente os mesmos valores em todas as variáveis. Também não foram encontradas duplicidades para a combinação **Customer_ID + Month**, indicando que cada cliente possui apenas um registro para cada mês analisado.

Foi identificado que a base contém aproximadamente **12.500 clientes distintos**, cada um com **8 registros** (100.000 ÷ 12.500 = 8). Essa característica confirma que a base possui uma estrutura temporal, permitindo acompanhar a evolução das informações financeiras dos clientes ao longo do tempo.

Esses resultados indicam boa consistência estrutural da base para as próximas etapas da análise.

### **Limpeza e Preparação dos Dados**